# 推論結果を読み込むノートブック

In [19]:
import torch
import os
import numpy as np
import pandas as pd
import glob
import copy

出力構造数は1110個。`evaluate.py`の`optimization`の`num_starting_points` x `num_saved_crys`. `num_saved_crys`は、最適化ステップ中に保存する数だと思う
```python
def optimization(model, ld_kwargs, data_loader,
                 num_starting_points=100, num_gradient_steps=5000,
                 lr=1e-3, num_saved_crys=10):
```

In [20]:
glob.glob('/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_*.pt')

['/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.01__grad-steps400.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.001__grad-steps1600.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr1e-05__grad-steps1600.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr1e-05__grad-steps800.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.0001__grad-steps3200.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.001__grad-steps5000.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.0001__grad-steps400.pt',
 '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__bg-1.00__lr0.0001__grad-steps5000.pt',
 '/home/f

In [21]:
result_dir = '/home/fujii/cdvae_comparison/main_results/cdvae_supercon'
os.makedirs(result_dir, exist_ok=True)

dict_keys(['frac_coords', 'atom_types', 'num_atoms', 'lengths', 'angles', 'prediction', 'prediction_decoded', 'prediction_matched', 'eval_setting', 'time'])

In [75]:
raw_result_dict_list = []
decoded_result_dict_list = []
for pred_path in glob.glob('/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/supercon/eval_opt_supercon__*.pt'):
    d = torch.load(pred_path)

    # ファイル名から学習率などを取得
    tag_dict = {}
    tag_name_list = ['lr','bg','grad-steps']
    for tag_data in os.path.basename(pred_path).split("__"):
        for tag_name in tag_name_list:
            if tag_data.startswith(tag_name):
                if tag_name == 'grad-steps':
                    tag_dict[tag_name] = int(tag_data.replace(tag_name,"").replace('.pt',''))
                elif tag_data.startswith('bg') or tag_data.startswith('eval_opt_supercon'):
                    pass
                else:
                    tag_dict[tag_name] = float(tag_data.replace(tag_name,"").replace('.pt',''))

    assert np.array(d['prediction_decoded']).shape[0] == np.array(d['prediction_matched']).shape[0]

    # 最適化直後のzから得た予測値
    raw_z_pred = np.array(d['prediction_matched']).squeeze()
    ## 128になるまで、np.nanを追加する
    if raw_z_pred.shape[0] < 128:
        raw_z_pred = np.concatenate([raw_z_pred, np.full((128 - raw_z_pred.shape[0]), np.nan)], axis=0)
    # 最適化後のzをdenoise-> crystal -> encode -> zで得た予測値
    decoded_z_pred = np.array(d['prediction_decoded']).squeeze()
    ## 128になるまで、np.nanを追加する
    if decoded_z_pred.shape[0] < 128:
        decoded_z_pred = np.concatenate([decoded_z_pred, np.full((128 - decoded_z_pred.shape[0]), np.nan)], axis=0)
    raw_result_dict = copy.deepcopy(tag_dict)
    decoded_result_dict =  copy.deepcopy(tag_dict)
    for tc in [2., 3., 4., 5., 7.5, 10., 12.5, 15., 20., 22.5, 25., 30., 40., 50.]:
        raw_result_dict[tc] = (raw_z_pred >tc).sum()/raw_z_pred.shape[0]
        decoded_result_dict[tc] = (decoded_z_pred >tc).sum()/decoded_z_pred.shape[0]
    raw_result_dict_list.append(raw_result_dict)
    decoded_result_dict_list.append(decoded_result_dict)
    
    if pd.DataFrame(decoded_z_pred).max().values > 2:
        display(pd.DataFrame(decoded_z_pred)[(pd.DataFrame(decoded_z_pred)>30).values])

/tmp/ipykernel_832520/437963749.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(pred_path)


,0
88,2185116.5


/tmp/ipykernel_832520/437963749.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(pred_path)


,0
56,331433.25


/tmp/ipykernel_832520/437963749.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(pred_path)


...

/tmp/ipykernel_832520/437963749.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(pred_path)


,0


,0


,0


,0


,0


In [32]:
raw_df = pd.DataFrame(raw_result_dict_list).set_index(['lr','grad-steps']).sort_index()
raw_df.to_csv(os.path.join(result_dir, 'raw_result.csv'))
raw_df

2.0       3.0       4.0       5.0       7.5  \
lr      grad-steps                                                     
0.00001 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.007812  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   
        5000        0.000000  0.000000  0.000000  0.000000  0.000000   
0.00010 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.007812  0.000000  0.000000  0.000000  0.000000   
        1600        0.007812  0.000000  0.000000  0.000000  0.000000   
        3200        0.070312  0.000000  0.000000  0.000000  0.000000   
        5000        0.101562  0.015625  0.000000  0.000000  0.000000   
0.00100 200         0.015625  0.000000  0.000000  0.000000  0.000000   
        400         0.078125  0.000000  0.000000  0.000000  0.000000   
        800         0.265625  0.125000  0.015625  0.000000  0.000000   
        1600        0.250000  0.250000  0.242188  0.203125  0.000000   
        3200        0.234375  0.234375  0.234375  0.234375  0.234375   
        5000        0.250000  0.250000  0.250000  0.250000  0.250000   
0.01000 200         0.250000  0.250000  0.250000  0.250000  0.015625   
        400         0.250000  0.250000  0.250000  0.250000  0.250000   
        800         0.085938  0.085938  0.085938  0.085938  0.085938   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   

                        10.0      12.5      15.0      20.0      22.5  \
lr      grad-steps                                                     
0.00001 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   
        5000        0.000000  0.000000  0.000000  0.000000  0.000000   
0.00010 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   
        5000        0.000000  0.000000  0.000000  0.000000  0.000000   
0.00100 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.203125  0.000000  0.000000  0.000000  0.000000   
        5000        0.250000  0.250000  0.234375  0.000000  0.000000   
0.01000 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.250000  0.218750  0.007812  0.000000  0.000000   
        800         0.085938  0.085938  0.085938  0.085938  0.085938   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   

                        25.0  30.0  40.0  50.0  
lr      grad-steps                              
0.00001 200         0.000000   0.0   0.0   0.0  
        400         0.000000   0.0   0.0   0.0  
        800         0.000000   0.0   0.0   0.0  
        1600        0.000000   0.0   0.0   0.0  
        3200        0.000000   0.0   0.0   0.0  
        5000        0.000000   0.0   0.0   0.0  
0.00010 200         0.000000   

In [33]:
decoded_df = pd.DataFrame(decoded_result_dict_list).set_index(['lr','grad-steps']).sort_index()
decoded_df.to_csv(os.path.join(result_dir, 'decoded_result.csv'))
decoded_df

2.0       3.0       4.0       5.0       7.5  \
lr      grad-steps                                                     
0.00001 200         0.007812  0.000000  0.000000  0.000000  0.000000   
        400         0.015625  0.007812  0.000000  0.000000  0.000000   
        800         0.046875  0.023438  0.000000  0.000000  0.000000   
        1600        0.015625  0.007812  0.000000  0.000000  0.000000   
        3200        0.023438  0.015625  0.000000  0.000000  0.000000   
        5000        0.007812  0.007812  0.000000  0.000000  0.000000   
0.00010 200         0.015625  0.007812  0.000000  0.000000  0.000000   
        400         0.023438  0.023438  0.000000  0.000000  0.000000   
        800         0.015625  0.015625  0.000000  0.000000  0.000000   
        1600        0.015625  0.000000  0.000000  0.000000  0.000000   
        3200        0.015625  0.007812  0.000000  0.000000  0.000000   
        5000        0.039062  0.000000  0.000000  0.000000  0.000000   
0.00100 200         0.015625  0.007812  0.000000  0.000000  0.000000   
        400         0.015625  0.000000  0.000000  0.000000  0.000000   
        800         0.039062  0.007812  0.000000  0.000000  0.000000   
        1600        0.039062  0.015625  0.007812  0.007812  0.007812   
        3200        0.039062  0.039062  0.031250  0.007812  0.007812   
        5000        0.015625  0.007812  0.007812  0.007812  0.007812   
0.01000 200         0.031250  0.015625  0.000000  0.000000  0.000000   
        400         0.023438  0.007812  0.007812  0.007812  0.007812   
        800         0.007812  0.007812  0.000000  0.000000  0.000000   
        1600        0.007812  0.007812  0.000000  0.000000  0.000000   
        3200        0.007812  0.000000  0.000000  0.000000  0.000000   

                        10.0      12.5      15.0      20.0      22.5  \
lr      grad-steps                                                     
0.00001 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   
        5000        0.000000  0.000000  0.000000  0.000000  0.000000   
0.00010 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   
        5000        0.000000  0.000000  0.000000  0.000000  0.000000   
0.00100 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.000000  0.000000  0.000000  0.000000  0.000000   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.007812  0.007812  0.007812  0.007812  0.007812   
        3200        0.007812  0.007812  0.007812  0.007812  0.007812   
        5000        0.007812  0.007812  0.007812  0.007812  0.007812   
0.01000 200         0.000000  0.000000  0.000000  0.000000  0.000000   
        400         0.007812  0.007812  0.007812  0.007812  0.007812   
        800         0.000000  0.000000  0.000000  0.000000  0.000000   
        1600        0.000000  0.000000  0.000000  0.000000  0.000000   
        3200        0.000000  0.000000  0.000000  0.000000  0.000000   

                        25.0      30.0      40.0      50.0  
lr      grad-steps                                          
0.00001 200         0.000000  0.000000  0.000000  0.000000  
        400         0.000000  0.000000  0.000000  0.000000  
        800         0.000000  0.000000  0.000000  0.000000  
        1600        0.000000  0.000000  0.000000  0.000000  
        3200        0.000000  0.000000  0.000000  0.00000

In [27]:
best_lr = 0.005
comparison_df = pd.concat(
    [raw_df.loc[[best_lr]].T.rename(columns={best_lr:'raw'}),
     decoded_df.loc[[best_lr]].T.rename(columns={best_lr:'decoded'})],
axis=1)
comparison_df.to_csv(os.path.join(result_dir, 'comparison_result.csv'))
comparison_df


KeyError: '[0.005] not in index'